# Paper Digest — MVP scaffold (Google Colab 版)

レポジトリを clone し、入力 PDF の一覧・サイズ・SHA-256 からコーパス manifest を作るスキャフォールドです。ページ抽出やグラフ・RAG は含みません（本番パイプラインは `paper_digest_colab.ipynb` を参照）。旧 Kaggle 版: `notebooks/kaggle/paper-digest-mvp.ipynb`。

In [ ]:
# Install the minimal dependencies
%pip -q install pandas numpy

## 0. Repository setup

In [ ]:
# Clone the canonical repository (re-run safe)
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/paper-digest-src')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/bonsai/paper-digest.git', str(REPO_DIR)], check=True)
print('Repo ready at', REPO_DIR)

## 1. Input folder

Colab では PDF を `/content` にアップロードするか、許諾済みの PDF を `inputs/literature/` に置いてください。**PDF はデフォルトでコミットしません**。アクセス・利用条件がこのワークフローを許すファイルだけを使います。

In [ ]:
UPLOAD_DIR = REPO_DIR / 'inputs' / 'literature'
FALLBACK_DIR = Path('/content/paper-pdfs')
FALLBACK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path('/content/paper-digest-output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: upload PDFs from your machine
# from google.colab import files
# files.upload()  # uploaded files land in /content; move them into the input folder

pdf_paths = sorted(UPLOAD_DIR.rglob('*.pdf')) + sorted(FALLBACK_DIR.rglob('*.pdf'))
print(f'PDFs found: {len(pdf_paths)}')
for path in pdf_paths:
    print('-', path.name)

if not pdf_paths:
    print('No PDFs found. Upload permitted PDFs or update INPUT_DIR.')

## 2. Corpus manifest and hashes

この手順はファイルの同一性を記録するだけです。PDF の解析済みであることや、コーパスが分野を代表することを主張するものではありません。

In [ ]:
import hashlib
import pandas as pd

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

manifest = pd.DataFrame([
    {'file_name': p.name, 'path': str(p), 'bytes': p.stat().st_size, 'sha256': sha256_file(p)}
    for p in pdf_paths
])
manifest.to_csv(OUTPUT_DIR / 'pdf_manifest.csv', index=False)
manifest

## Next implementation steps

- ページ単位の抽出とファイルごとのエラー報告は `paper_digest_colab.ipynb`（共有パイプライン）で実装済み。
- papers / chunks のスキーマは `outputs/schema.json` を参照。
- グラフのエッジは共起・引用・意味的類似・データセット・指標を分離する。
- 固定質問セットについてページ単位の引用を人手で検証する。
- 実行ごとに依存バージョン・コーパスハッシュ・制約を記録する。